# 13 — Export, backup and restore

## Goal
Package an explicitly selected trained AHR Go checkpoint together with all 14 source notebooks, the retained latest two checkpoints and best checkpoint, exact recovery inputs, and a version inventory. Verify every byte, restore into a new isolated directory, then load and run the restored player in a fresh CPU process.

**Default Run All:** local export and restore checks only; no training and no upload. The selected warmup is a candidate whose playing strength has not been established. A verified local archive is **not an off-instance backup**.

**Inputs:** notebooks 00–13, a committed Go checkpoint manifest, its retained payloads and recovery data. **Outputs:** `exports/bundles/`, `exports/restored/`, `reports/13_export_restore_validation.json`. **Expected runtime:** 1–3 minutes for the small warmup; large self-play replay buffers take longer. **Checkpoint location:** `checkpoints/<run_id>/` is copied intact into the restored workspace. This notebook does not train or change the source run.

**Resume:** rerunning Run All creates a new bundle and isolated restore; it never overwrites an existing restore. For training continuation, use the restored `bundle/resume.json` and the original notebook 08 or 09 instructions below. Invalid/missing manifests fail explicitly.

## Setup
Only `ahr-export` definition cells are imported. The model loader imports 00–06 and 08; no upstream generation, training or notebook workflow cells are executed. Choose `AHR13_RUN_ID` and `AHR13_SELECTION` before launching the kernel to package a different completed/paused run. An active checkpoint writer causes export to stop rather than copy a moving run.

For the final self-play candidate use `AHR13_RUN_ID=go09-first-selfplay-v1` and `AHR13_SELECTION=latest`. Optionally set `AHR13_EXTRA_FILES` to a JSON list of exact companion/report paths; missing files stop the export. For example, include `reports/09_first_selfplay_run.ipynb` and `reports/11_first_candidate_evaluation.ipynb` once both exist. No upload is enabled by these settings.

In [ ]:
from pathlib import Path
import json, os, sys, hashlib, io, math, re, shutil, subprocess, tarfile, tempfile, uuid, platform
import importlib.metadata
from datetime import datetime, timezone
from pathlib import PurePosixPath
import nbformat
import torch

PROJECT = Path.cwd().resolve()
if not (PROJECT / "notebooks").is_dir() and (PROJECT.parent / "notebooks").is_dir():
    PROJECT = PROJECT.parent
if not (PROJECT / "notebooks/00_instance_setup_and_recovery.ipynb").is_file():
    raise RuntimeError("Start this kernel in the AHR Go workspace or notebooks directory")
for number in (*range(7), 8):
    paths = list((PROJECT / "notebooks").glob(f"{number:02d}_*.ipynb"))
    if len(paths) != 1:
        raise ValueError(f"Expected exactly one notebook {number:02d}")
    for index, cell in enumerate(nbformat.read(paths[0], as_version=4).cells):
        if cell.cell_type == "code" and "ahr-export" in cell.metadata.get("tags", []):
            exec(compile(cell.source, str(paths[0]) + f":export-{index}", "exec"), globals())
torch.set_num_threads(1)
RUN_ID = os.environ.get("AHR13_RUN_ID", "go08-initial-warmup-v2")
SELECTION = os.environ.get("AHR13_SELECTION", "best")
EXTRA_FILES = json.loads(os.environ.get("AHR13_EXTRA_FILES", "[]"))
if not isinstance(EXTRA_FILES, list) or any(not isinstance(x, str) for x in EXTRA_FILES):
    raise ValueError("AHR13_EXTRA_FILES must be a JSON list of relative paths")
print(f"Selected run: {RUN_ID}; checkpoint: {SELECTION}; local validation only")

## Package a fixed checkpoint
The archive contains regular files only. A SHA-256 inventory binds every file, and a separate archive digest detects any change to the entire archive. The original checkpoint manifest is preserved, including the latest two valid snapshots and best. Warmup data and recovery receipts are included; self-play state already embeds replay, current game, opponent snapshots and random states. Notebook outputs are cleared in the portable source copy so execution history cannot be mistaken for a new run.

Hashes detect corruption when the expected digest is obtained from a trusted source; they do not authenticate an untrusted author. Loading a PyTorch checkpoint or executing notebook definitions is reserved for these locally produced, trusted bundles.

Generated PNG report figures may be included as extra files; they receive the same inventory hash and restore checks as other report artifacts.

In [ ]:
# Exported definitions carry their own standard-library imports.
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
import json, os, sys, hashlib, io, re, shutil, subprocess, tarfile, tempfile, uuid, platform
import importlib.metadata
import nbformat
import torch

BUNDLE13_SCHEMA = "ahr_go_portable_bundle_v1"

def bundle13_json_bytes(value):
    return (json.dumps(value, sort_keys=True, indent=2, ensure_ascii=False) + "\n").encode()

def bundle13_name(value):
    if not isinstance(value, str) or not value or not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]{0,160}", value):
        raise ValueError("Use a simple nonempty name")
    return value

def bundle13_relative(value):
    if not isinstance(value, str) or not value or "\\" in value or any(ord(c) < 32 for c in value):
        raise ValueError("Unsafe archive path")
    parts = value.split("/")
    if any(p in ("", ".", "..") for p in parts) or PurePosixPath(value).is_absolute() or ":" in parts[0]:
        raise ValueError("Unsafe archive path")
    return PurePosixPath(value)

def bundle13_source_file(project, relative):
    project = Path(project).resolve(); relative = bundle13_relative(relative)
    candidate = project.joinpath(*relative.parts)
    if any(part.is_symlink() for part in [candidate, *candidate.parents] if part != project.parent):
        raise ValueError("Symlink input refused")
    if not candidate.is_file() or not candidate.resolve().is_relative_to(project):
        raise ValueError("Missing or out-of-workspace input: " + str(relative))
    return candidate

def bundle13_notebook_bytes(path):
    book = nbformat.read(path, as_version=4); nbformat.validate(book)
    for cell in book.cells:
        if cell.cell_type == "code":
            cell.outputs = []; cell.execution_count = None
    return nbformat.writes(book).encode()

def bundle13_environment():
    return {"python": platform.python_version(), "platform": platform.platform(),
            "torch": str(torch.__version__), "torch_cuda_build": torch.version.cuda,
            "packages": dict(sorted((d.metadata["Name"], d.version) for d in importlib.metadata.distributions() if d.metadata.get("Name"))),
            "recovery_requirement": "Match recorded packages, GPU model/count, device mapping and sources for exact training continuation; CPU inference is tested separately."}

def bundle13_write_archive(path, files):
    path = Path(path)
    with open(path, "xb") as raw:
        with tarfile.open(fileobj=raw, mode="w:gz", format=tarfile.PAX_FORMAT) as archive:
            for name, content in sorted(files.items()):
                bundle13_relative(name)
                entry = tarfile.TarInfo(name); entry.size = len(content); entry.mode = 0o600; entry.mtime = 0
                archive.addfile(entry, io.BytesIO(content))
        raw.flush(); os.fsync(raw.fileno())

def export_go_bundle(project, run_id, selection="best", notebook_numbers=tuple(range(14)), extra_files=()):
    """Create an immutable local snapshot; never start or resume a trainer."""
    project = Path(project).resolve(); run_id = bundle13_name(run_id)
    if selection not in ("latest", "best"):
        raise ValueError("Select latest or best")
    if set(notebook_numbers) != set(range(14)) or len(notebook_numbers) != 14:
        raise ValueError("A complete handoff requires exactly notebooks 00–13")
    run_dir = project / "checkpoints" / run_id
    if not run_dir.is_dir() or run_dir.is_symlink():
        raise ValueError("Selected checkpoint run does not exist or is a symlink")
    files = {}
    # A brief shared checkpoint lock avoids copying a moving writer's files.
    with checkpoint_lock(run_dir):
        latest = strict_go_checkpoint(project, run_id, "latest")
        selected = strict_go_checkpoint(project, run_id, selection)
        if latest is None or selected is None:
            raise ValueError("Selected run has no committed manifest")
        model, metadata = load_go_ahr(project, run_id, "cpu", selection=selection)
        manifest_path = run_dir / "manifest.json"
        checkpoint_manifest = json.loads(manifest_path.read_text())
        entries = sorted(checkpoint_manifest["entries"], key=lambda item: item["global_step"])
        if len(entries) < 2:
            raise ValueError("A continuation bundle requires at least two valid retained snapshots")
        retained = {entry["file"] for entry in entries}
        if checkpoint_manifest.get("best") not in retained or checkpoint_manifest["latest"] != entries[-1]["file"]:
            raise ValueError("Missing best/latest retention")
        for number in notebook_numbers:
            paths = list((project / "notebooks").glob(f"{number:02d}_*.ipynb"))
            if len(paths) != 1:
                raise ValueError(f"Expected exactly one notebook {number:02d}; finish authoring before export")
            relative = paths[0].relative_to(project).as_posix()
            files[relative] = bundle13_notebook_bytes(bundle13_source_file(project, relative))
        snapshot_hashes = {}
        for name in latest["source_hashes"]:
            book = nbformat.reads(files["notebooks/" + name].decode(), as_version=4)
            snapshot_hashes[name] = hashlib.sha256(json.dumps([c.source for c in book.cells if c.cell_type == "code" and "ahr-export" in c.metadata.get("tags", [])], ensure_ascii=False).encode()).hexdigest()
        if snapshot_hashes != latest["source_hashes"]:
            raise ValueError("Snapshot source definitions differ from the checkpoint")
        if latest["schema"] == "ahr_go_selfplay_v1":
            book = nbformat.reads(files["notebooks/09_selfplay_and_improvement.ipynb"].decode(), as_version=4)
            digest = hashlib.sha256("\n\n".join(c.source for c in book.cells if c.cell_type == "code" and "ahr-export" in c.metadata.get("tags", [])).encode()).hexdigest()
            if digest != latest.get("selfplay_source_hash"):
                raise ValueError("Self-play source differs from checkpoint")
        for relative in extra_files:
            source = bundle13_source_file(project, relative)
            if relative in files or relative.startswith(("checkpoints/", "bundle/")) or relative == "bundle_manifest.json":
                raise ValueError("Extra files cannot replace bundle inputs")
            if source.suffix not in (".ipynb", ".json", ".md", ".txt", ".log", ".png"):
                raise ValueError("Extra inputs must be notebook, documentation or report files")
            files[relative] = bundle13_notebook_bytes(source) if source.suffix == ".ipynb" else source.read_bytes()
        for entry in entries:
            relative = f"checkpoints/{run_id}/{entry['file']}"
            content = bundle13_source_file(project, relative).read_bytes()
            if hashlib.sha256(content).hexdigest() != entry["sha256"]:
                raise ValueError("Checkpoint changed during snapshot")
            files[relative] = content
        files[f"checkpoints/{run_id}/manifest.json"] = manifest_path.read_bytes()
        if latest["schema"] == "ahr_go_learning_v1":
            data_file = latest["config"]["data_file"]
            files[data_file] = bundle13_source_file(project, data_file).read_bytes()
            if hashlib.sha256(files[data_file]).hexdigest() != latest["config"]["data_sha256"]:
                raise ValueError("Warmup data changed")
            receipt = "reports/08_exact_resume_validation.json"
            files[receipt] = bundle13_source_file(project, receipt).read_bytes()
        elif latest["schema"] == "ahr_go_selfplay_v1":
            receipts = sorted((project / "reports/09_recovery_receipts").glob("*.json"))
            if not receipts:
                raise ValueError("Missing self-play recovery receipts")
            for receipt in receipts:
                relative = receipt.relative_to(project).as_posix()
                files[relative] = bundle13_source_file(project, relative).read_bytes()
        else:
            raise ValueError("Unsupported training checkpoint schema")
        for relative in ("AGENTS.md", "README.md"):
            files[relative] = bundle13_source_file(project, relative).read_bytes()
        files["bundle/environment.json"] = bundle13_json_bytes(bundle13_environment())
        resume = {"schema": latest["schema"], "run_id": run_id, "resume_selection": "latest", "inference_selection": selection,
                  "latest_optimizer_step": latest["optimizer_step"], "selected_optimizer_step": selected["optimizer_step"],
                  "global_step": latest["global_step"], "config": latest["config"], "model_config": latest["model_config"],
                  "resume_cell": checkpoint_manifest["resume_cell"], "source_hashes": latest["source_hashes"],
                  "stage": latest.get("stage", latest.get("phase")), "checkpoint_keys": sorted(latest),
                  "identity": latest.get("identity"), "selfplay_source_hash": latest.get("selfplay_source_hash"),
                  "retained_checkpoint_files": sorted(retained), "exact_training_resume_executed_here": False,
                  "restore_validation": "Checkpoint bytes, source definitions and recovery inputs; fresh CPU inference. Existing source-bound training recovery receipts are preserved."}
        files["bundle/resume.json"] = bundle13_json_bytes(resume)
        files["bundle/RESTORE.md"] = b"Restore only this trusted archive with notebook 13 and its separately verified SHA-256. Run tagged definition loaders, not upstream Run All. Read environment.json and resume.json before training continuation. Match original devices, sources and package versions. Current warmup may already be complete; do not silently reset its run ID. Off-instance backup is separate.\n"
        if latest["source_hashes"] != go08_source_hashes(project):
            raise ValueError("Architecture sources changed during snapshot")
        inventory = {name: {"sha256": hashlib.sha256(content).hexdigest(), "bytes": len(content)} for name, content in sorted(files.items())}
        bundle_id = run_id + "-" + uuid.uuid4().hex[:12]
        manifest = {"schema": BUNDLE13_SCHEMA, "bundle_id": bundle_id,
                    "created_utc": datetime.now(timezone.utc).isoformat(), "checkpoint": metadata,
                    "retention": "All committed retained payloads, including latest two and best", "files": inventory,
                    "notebook_count": 14, "strength_status": "NOT_ESTABLISHED_BY_EXPORT", "off_instance_backup": "PENDING"}
        files["bundle_manifest.json"] = bundle13_json_bytes(manifest)
        target = project / "exports/bundles"; target.mkdir(parents=True, exist_ok=True)
        archive = target / (bundle_id + ".tar.gz")
        temporary = target / ("." + bundle_id + ".partial")
        try:
            bundle13_write_archive(temporary, files)
            digest = file_sha256(temporary)
            os.replace(temporary, archive)
            atomic_write_bytes(archive.with_suffix(archive.suffix + ".sha256"), (digest + "  " + archive.name + "\n").encode())
        finally:
            temporary.unlink(missing_ok=True)
    return {"archive": str(archive), "sha256": digest, "bundle_id": bundle_id,
            "bytes": archive.stat().st_size, "file_count": len(files), "checkpoint": metadata,
            "retained_checkpoints": len(entries), "off_instance_backup": "PENDING"}


## Verify and restore safely
Restoration requires an expected archive SHA-256 from the producer. Before extraction, inspect all members and the complete inventory: reject absolute paths, traversal, links, special files, duplicate entries, unexpected/missing files, size changes and hash failures. Restore into a newly created staging directory, then rename it atomically to a new directory under `exports/restored/`; existing files are never replaced. The default uncompressed size limit is 20 GiB.

In [ ]:
def verify_go_bundle(archive_path, expected_sha256, max_uncompressed_bytes=20 * 1024**3):
    archive_path = Path(archive_path)
    if not re.fullmatch(r"[0-9a-f]{64}", str(expected_sha256)) or file_sha256(archive_path) != expected_sha256:
        raise ValueError("Archive SHA-256 mismatch")
    with tarfile.open(archive_path, "r:gz") as archive:
        members = archive.getmembers()
        names = [item.name for item in members]
        if len(names) != len(set(names)):
            raise ValueError("Duplicate archive member")
        total = 0
        for item in members:
            bundle13_relative(item.name)
            if not item.isreg() or item.size < 0:
                raise ValueError("Only regular archive files are allowed")
            total += item.size
        if total > max_uncompressed_bytes:
            raise ValueError("Archive exceeds extraction size limit")
        if "bundle_manifest.json" not in names:
            raise ValueError("Missing bundle manifest")
        manifest_entry = archive.getmember("bundle_manifest.json")
        if manifest_entry.size > 8 * 1024**2:
            raise ValueError("Oversized bundle manifest")
        manifest = json.load(archive.extractfile(manifest_entry))
        if manifest.get("schema") != BUNDLE13_SCHEMA or not isinstance(manifest.get("files"), dict):
            raise ValueError("Invalid bundle manifest")
        bundle13_name(manifest.get("bundle_id"))
        expected = manifest["files"]
        if set(names) != set(expected) | {"bundle_manifest.json"} or "bundle_manifest.json" in expected:
            raise ValueError("Missing or unexpected archive files")
        for name, entry in expected.items():
            relative = bundle13_relative(name)
            if any(str(parent) in expected for parent in relative.parents if str(parent) != "."):
                raise ValueError("File/directory path collision")
            if not isinstance(entry, dict) or type(entry.get("bytes")) is not int or entry["bytes"] < 0 or not re.fullmatch(r"[0-9a-f]{64}", str(entry.get("sha256"))):
                raise ValueError("Invalid file inventory")
            item = archive.getmember(name)
            if item.size != entry["bytes"]:
                raise ValueError("Inventory size mismatch")
            digest = hashlib.sha256()
            with archive.extractfile(item) as stream:
                for block in iter(lambda: stream.read(1024**2), b""):
                    digest.update(block)
            if digest.hexdigest() != entry["sha256"]:
                raise ValueError("Inventory digest mismatch")
        checkpoint = manifest.get("checkpoint", {})
        run_id = bundle13_name(checkpoint.get("run_id"))
        checkpoint_manifest_name = f"checkpoints/{run_id}/manifest.json"
        if checkpoint_manifest_name not in expected:
            raise ValueError("Missing checkpoint manifest")
        committed = json.load(archive.extractfile(checkpoint_manifest_name))
        entries = committed.get("entries", [])
        if committed.get("schema_version") != 1 or len(entries) < 2:
            raise ValueError("Invalid checkpoint retention manifest")
        if any(not isinstance(e, dict) or type(e.get("global_step")) is not int or e["global_step"] < 0 for e in entries):
            raise ValueError("Invalid checkpoint event")
        if len({e.get("file") for e in entries}) != len(entries) or len({e["global_step"] for e in entries}) != len(entries):
            raise ValueError("Duplicate checkpoint entry")
        if committed.get("latest") != max(entries, key=lambda e: e["global_step"])["file"] or committed.get("best") not in {e["file"] for e in entries}:
            raise ValueError("Invalid checkpoint latest/best")
        for entry in entries:
            filename = bundle13_name(entry.get("file"))
            name = f"checkpoints/{run_id}/{filename}"
            if name not in expected or expected[name]["sha256"] != entry.get("sha256"):
                raise ValueError("Checkpoint payload not bound by both manifests")
        if checkpoint.get("checkpoint_file") not in expected or expected[checkpoint["checkpoint_file"]]["sha256"] != checkpoint.get("checkpoint_sha256"):
            raise ValueError("Selected model is not bound by manifest")
        if manifest.get("notebook_count") != 14 or any(len([n for n in expected if re.fullmatch(f"notebooks/{i:02d}_[^/]+\\.ipynb", n)]) != 1 for i in range(14)):
            raise ValueError("Incomplete source notebook set")
    return {"status": "PASS", "manifest": manifest, "files": len(members), "uncompressed_bytes": total}

def restore_go_bundle(archive_path, project, restore_id, expected_sha256):
    project = Path(project).resolve(); restore_id = bundle13_name(restore_id)
    verified = verify_go_bundle(archive_path, expected_sha256)
    parent = project / "exports/restored"
    parent.mkdir(parents=True, exist_ok=True)
    if parent.is_symlink() or not parent.resolve().is_relative_to(project):
        raise ValueError("Restore root must be inside the workspace")
    destination = parent / restore_id
    if destination.exists() or destination.is_symlink():
        raise FileExistsError("Restore destination already exists; choose a new restore ID")
    if shutil.disk_usage(parent).free < verified["uncompressed_bytes"] + 64 * 1024**2:
        raise ValueError("Insufficient free space for isolated restore")
    staging = Path(tempfile.mkdtemp(prefix=".restore-", dir=parent))
    try:
        with tarfile.open(archive_path, "r:gz") as archive:
            for entry in archive.getmembers():
                target = staging.joinpath(*bundle13_relative(entry.name).parts)
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.extractfile(entry) as source, open(target, "xb") as output:
                    shutil.copyfileobj(source, output)
                    output.flush(); os.fsync(output.fileno())
                target.chmod(0o600)
        # Recheck bytes after copy, before making the restored workspace visible.
        for relative, entry in verified["manifest"]["files"].items():
            if file_sha256(staging / relative) != entry["sha256"]:
                raise ValueError("Restored file digest mismatch")
        if file_sha256(archive_path) != expected_sha256:
            raise ValueError("Archive changed during restoration")
        if destination.exists() or destination.is_symlink():
            raise FileExistsError("Restore destination appeared during extraction")
        os.rename(staging, destination)
    finally:
        if staging.exists():
            shutil.rmtree(staging)
    return {"status": "PASS", "workspace": str(destination), "files": verified["files"],
            "sha256": expected_sha256, "off_instance_backup": "PENDING"}


## Optional off-instance copy
No destination is guessed from credentials. `copy_go_bundle_off_instance(archive, digest, "gs://bucket/explicit-prefix")` is an explicit operation: it uploads the archive and digest, downloads the uploaded archive to a temporary local directory, and verifies its SHA-256. Keep that receipt before recycling/deleting the VM. Without an authorized destination and successful remote verification, off-instance durability remains **PENDING**. The default workflow never calls this function.

In [ ]:
def copy_go_bundle_off_instance(archive_path, expected_sha256, destination):
    from urllib.parse import urlsplit
    archive_path = Path(archive_path).resolve()
    verify_go_bundle(archive_path, expected_sha256)
    uri = urlsplit(destination)
    if uri.scheme != "gs" or not re.fullmatch(r"[a-z0-9][a-z0-9._-]{1,220}[a-z0-9]", uri.netloc) or uri.query or uri.fragment:
        raise ValueError("Provide an explicit gs://bucket/prefix destination")
    prefix = uri.path.strip("/")
    if prefix:
        bundle13_relative(prefix)
    if shutil.which("gsutil") is None:
        raise RuntimeError("Install/authenticate Google Cloud CLI gsutil for this explicit destination")
    remote = "gs://" + uri.netloc + ("/" + prefix if prefix else "") + "/" + archive_path.name
    # UUID bundle names make this an immutable upload; refuse a known existing object.
    existing = subprocess.run(["gsutil", "-q", "stat", remote], capture_output=True, text=True)
    if existing.returncode == 0:
        raise FileExistsError("Remote object already exists; export a new bundle")
    sidecar = archive_path.with_suffix(archive_path.suffix + ".sha256")
    atomic_write_bytes(sidecar, (expected_sha256 + "  " + archive_path.name + "\n").encode())
    for source, target in ((archive_path, remote), (sidecar, remote + ".sha256")):
        subprocess.run(["gsutil", "-q", "cp", str(source), target], check=True, capture_output=True, text=True)
    with tempfile.TemporaryDirectory(prefix="13-remote-verify-", dir=archive_path.parent) as temporary:
        download = Path(temporary) / archive_path.name
        subprocess.run(["gsutil", "-q", "cp", remote, str(download)], check=True, capture_output=True, text=True)
        if file_sha256(download) != expected_sha256:
            raise ValueError("Remote roundtrip SHA-256 verification failed")
    receipt = {"status": "PASS", "off_instance_backup": "VERIFIED", "destination": remote,
               "sha256": expected_sha256, "verified_utc": datetime.now(timezone.utc).isoformat(),
               "verification": "Downloaded remote archive and matched full SHA-256", "retention_policy": "NOT_VERIFIED"}
    atomic_json(archive_path.with_suffix(archive_path.suffix + ".remote.json"), receipt)
    return receipt

BUNDLE13_INFERENCE_WORKER = r"""
import json, pathlib, sys, nbformat, torch
project = pathlib.Path(sys.argv[1]); run_id = sys.argv[2]; selection = sys.argv[3]
for number in (*range(7), 8):
    paths = list((project / 'notebooks').glob(f'{number:02d}_*.ipynb'))
    if len(paths) != 1: raise ValueError('Incomplete restored notebook set')
    for index, cell in enumerate(nbformat.read(paths[0], 4).cells):
        if cell.cell_type == 'code' and 'ahr-export' in cell.metadata.get('tags', []):
            exec(compile(cell.source, str(paths[0]) + ':export-' + str(index), 'exec'), globals())
torch.set_num_threads(1)
model, metadata = load_go_ahr(project, run_id, device='cpu', selection=selection)
latest = strict_go_checkpoint(project, run_id, 'latest')
state = initial_state(9, 7.5)
result = choose_go_ahr_move(state, model, time_budget_s=2.0, search_config=GoAHRSearchConfig(max_depth=1, branch_width=2, max_nodes=4, queen_cycles=1))
assert result['action'] in legal_actions(state)
assert verify_go_emission(state, result['action'], play(state, result['action']))
print(json.dumps({'status':'PASS', 'device':'cpu', 'fresh_process':True, 'checkpoint_sha256':metadata['checkpoint_sha256'],
                  'model_sha256':metadata['model_sha256'], 'selected_optimizer_step':metadata['optimizer_step'],
                  'latest_optimizer_step':latest['optimizer_step'], 'action':result['action'], 'legal':True,
                  'strategic_status':result['strategic_status'], 'elapsed_s':result['elapsed_s']}))
"""

def validate_restored_go_inference(restored_project, run_id, selection="best"):
    result = subprocess.run([sys.executable, "-c", BUNDLE13_INFERENCE_WORKER, str(restored_project), run_id, selection],
                            cwd=restored_project, capture_output=True, text=True, timeout=120,
                            env={**os.environ, "CUDA_VISIBLE_DEVICES": "", "OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1"})
    if result.returncode:
        raise RuntimeError("Fresh restored inference failed: " + result.stderr[-2500:])
    return json.loads(result.stdout.strip().splitlines()[-1])


## Checks
These disposable archive fixtures deliberately remove or corrupt manifests, alter payload bytes, and add traversal/link/duplicate members. Each must fail **before any checkpoint is loaded**. The real selected bundle is then restored and tested in a fresh process with CUDA hidden. This checks portable inference and preservation of recovery state; it does not rerun training or claim cross-hardware bitwise optimizer recovery.

In [ ]:
def validate_bundle13_rejections(project, bundle):
    project = Path(project); passed = []
    with tarfile.open(bundle["archive"], "r:gz") as archive:
        original = {member.name: archive.extractfile(member).read() for member in archive.getmembers()}
    with tempfile.TemporaryDirectory(prefix="13-corruption-", dir=project / "exports") as temporary:
        temporary = Path(temporary)
        def expect_failure(name, files=None, special=None, wrong_digest=False):
            path = temporary / (name + ".tar.gz")
            if special:
                with tarfile.open(path, "w:gz") as archive:
                    for member_name, content in original.items():
                        entry = tarfile.TarInfo(member_name); entry.size = len(content)
                        archive.addfile(entry, io.BytesIO(content))
                    archive.addfile(*special)
            else:
                bundle13_write_archive(path, files)
            try:
                verify_go_bundle(path, "0" * 64 if wrong_digest else file_sha256(path))
            except (ValueError, KeyError, TypeError, json.JSONDecodeError):
                passed.append(name)
            else:
                raise AssertionError("Invalid archive accepted: " + name)
        missing = dict(original); missing.pop("bundle_manifest.json")
        expect_failure("missing_bundle_manifest", missing)
        corrupt = dict(original); corrupt["bundle_manifest.json"] = b"not json"
        expect_failure("corrupt_bundle_manifest", corrupt)
        changed = dict(original); data_name = next(name for name in changed if name.endswith(".pt")); changed[data_name] += b"changed"
        expect_failure("corrupt_payload", changed)
        missing_checkpoint = dict(original)
        manifest_name = "checkpoints/" + bundle["checkpoint"]["run_id"] + "/manifest.json"; missing_checkpoint.pop(manifest_name)
        inventory = json.loads(missing_checkpoint["bundle_manifest.json"]); inventory["files"].pop(manifest_name)
        missing_checkpoint["bundle_manifest.json"] = bundle13_json_bytes(inventory)
        expect_failure("missing_checkpoint_manifest", missing_checkpoint)
        corrupt_checkpoint = dict(original); committed = json.loads(corrupt_checkpoint[manifest_name]); committed["latest"] = "missing.pt"
        corrupt_checkpoint[manifest_name] = bundle13_json_bytes(committed)
        inventory = json.loads(corrupt_checkpoint["bundle_manifest.json"])
        inventory["files"][manifest_name] = {"bytes":len(corrupt_checkpoint[manifest_name]), "sha256":hashlib.sha256(corrupt_checkpoint[manifest_name]).hexdigest()}
        corrupt_checkpoint["bundle_manifest.json"] = bundle13_json_bytes(inventory)
        expect_failure("corrupt_checkpoint_manifest", corrupt_checkpoint)
        expect_failure("wrong_archive_digest", dict(original), wrong_digest=True)
        for name, member_name in (("parent_traversal", "../escape"), ("absolute_path", "/tmp/escape")):
            member = tarfile.TarInfo(member_name); member.size = 1
            expect_failure(name, special=(member, io.BytesIO(b"x")))
        for name, kind in (("symlink", tarfile.SYMTYPE), ("hardlink", tarfile.LNKTYPE)):
            member = tarfile.TarInfo("escape-link"); member.type = kind; member.linkname = "../escape"
            expect_failure(name, special=(member, None))
        member = tarfile.TarInfo("bundle_manifest.json"); member.size = 0
        expect_failure("duplicate_member", special=(member, io.BytesIO(b"")))
        member = tarfile.TarInfo("unlisted-file"); member.size = 1
        expect_failure("unexpected_member", special=(member, io.BytesIO(b"x")))
        try:
            verify_go_bundle(bundle["archive"], bundle["sha256"], max_uncompressed_bytes=1)
        except ValueError:
            passed.append("size_limit")
        else:
            raise AssertionError("Extraction size limit ignored")
    return {"status": "PASS", "checks": passed, "count": len(passed)}


### Run the local handoff
The source run is preserved. The receipt names the selected artifact, the archive digest, the restored workspace, all rejection checks and the exact inference result. The local filesystem report does not establish a cloud disk deletion policy.

In [ ]:
bundle = export_go_bundle(PROJECT, RUN_ID, SELECTION, extra_files=EXTRA_FILES)
verified = verify_go_bundle(bundle["archive"], bundle["sha256"])
restored = restore_go_bundle(bundle["archive"], PROJECT, bundle["bundle_id"], bundle["sha256"])
rejections = validate_bundle13_rejections(PROJECT, bundle)
inference = validate_restored_go_inference(restored["workspace"], RUN_ID, SELECTION)
assert inference["checkpoint_sha256"] == bundle["checkpoint"]["checkpoint_sha256"]
assert inference["model_sha256"] == bundle["checkpoint"]["model_sha256"]
try:
    restore_go_bundle(bundle["archive"], PROJECT, bundle["bundle_id"], bundle["sha256"])
except FileExistsError:
    destination_reuse_rejected = True
else:
    raise AssertionError("Existing restore was overwritten")
mount = subprocess.run(["findmnt", "-T", str(PROJECT), "-n", "-o", "TARGET,SOURCE,FSTYPE"], capture_output=True, text=True)
receipt = {"status":"PASS", "created_utc":datetime.now(timezone.utc).isoformat(), "bundle":bundle,
           "restored":restored, "rejections":rejections, "inference":inference,
           "existing_destination_rejected":destination_reuse_rejected,
           "storage":{"mount":mount.stdout.strip() if mount.returncode == 0 else "UNVERIFIED",
                      "instance_deletion_policy":"UNVERIFIED", "off_instance_backup":"PENDING", "authorized_destination":None},
           "training_launched":False, "graphical_review":"NOT_PERFORMED; inspect generated HTML in a browser",
           "source_exports_sha256":hashlib.sha256("\n\n".join(c.source for c in nbformat.read(PROJECT / "notebooks/13_export_backup_and_restore.ipynb",4).cells if c.cell_type=="code" and "ahr-export" in c.metadata.get("tags",[])).encode()).hexdigest()}
atomic_json(PROJECT / "reports/13_export_restore_validation.json", receipt)
print(json.dumps({"status":receipt["status"], "archive":str(Path(bundle["archive"]).relative_to(PROJECT)),
                  "archive_sha256":bundle["sha256"], "notebooks":14, "retained_checkpoints":bundle["retained_checkpoints"],
                  "restore":str(Path(restored["workspace"]).relative_to(PROJECT)), "rejection_checks":rejections["count"],
                  "fresh_cpu_inference":inference, "off_instance_backup":"PENDING"}, indent=2))

## Exact continuation and handoff

1. Keep the archive and its `.sha256` sidecar together. Obtain the expected digest through the same trusted handoff as the archive. Use the restore function above with a **new** restore ID; do not unpack over a live workspace.
2. Read `bundle/environment.json` in the restored workspace. Install a Python environment matching the recorded Python/PyTorch build and dependencies, including `torch`, `numpy`, `nbformat`, `nbclient`, `ipykernel` and `ipywidgets`. Install the matching PyTorch build from the official PyTorch package channel; package version inventory is included, but machine-specific wheels and the virtual environment are not embedded. Register the project kernel as `ahr-go`. CPU inference works without GPUs; exact GPU training continuation requires the recorded GPU/device mapping and compatible deterministic runtime.
3. For **08**, import the tagged definitions 00–06 and 08 and read `bundle/resume.json`. Call `run_go_warmup(restored_project, resume["run_id"], resume["config"]["data_file"], GoWarmupConfig(**resume["config"]["train"]), GoAHRConfig(**resume["model_config"]))`. The original run ID and identity load **latest**, including optimizer, scheduler, scaler, all RNG states, epoch/order/cursor and metrics. A completed warmup returns complete; additional training requires an explicit new run/design rather than changing the committed identity.
4. For **09**, import tagged definitions 00–06, 08 and 09. Call `run_selfplay_training(restored_project, resume["run_id"], GoAHRConfig(**resume["model_config"]), SelfPlayConfig(**resume["config"]), allow_long_run=True)`. Preserved source-bound recovery receipts are required; the run restores latest replay, current game/history, actor/learner state, opponent pool, sampler, counters and all RNGs. Do not pass a new initialization model while resuming. If runtime/device configuration changes, perform the notebook 09 recovery validation for that environment before continuing; do not alter the existing run identity to silence a mismatch.
5. Inspect the resumed step/metrics before allowing more work. For play only, use notebook 12 or 10 with the restored project and selected run; `best` and `latest` are explicit, separate selections. Exporting a candidate does not promote it or establish strong play.
6. After a destination has been authorized, call the optional upload function and keep its remote roundtrip receipt. **Until that passes, the VM must not be recycled/deleted on the assumption this local archive is an off-instance backup.**

## Result
The executed cells above provide the local export/restore result and CPU inference evidence. Training recovery state is preserved byte-for-byte; this notebook runs no optimizer steps. Graphical presentation review requires opening the generated HTML preview in a browser; successful execution alone is not a visual review.